# Semantic Search

**Multilingual Sentence Transformer-Based Semantic Retrieval**

This notebook implements semantic similarity retrieval for graduation projects using a pretrained multilingual Sentence Transformer and Cosine Similarity.

The system supports Arabic and English project ideas and retrieves the most semantically related historical projects.

## 1. Setup and Dependencies

Install and import the required libraries for semantic retrieval.

In [1]:
%pip install -q sentence-transformers

In [2]:
import pandas as pd
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

print("Libraries imported successfully!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Libraries imported successfully!
Device: cuda


## 2. Load and Validate Dataset

Load the cleaned graduation projects dataset from GitHub and verify its structure.

In [5]:
# Load dataset
url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/processed/projects_clean.csv"
)

df = pd.read_csv(url, encoding="utf-8-sig")

# Standardize columns
df.columns = df.columns.str.strip().str.lower()
df["department"] = df["department"].str.strip()

# Validate dataset
assert df.shape == (103, 7)
assert df["project_id"].is_unique
assert df[["title", "abstract"]].notna().all().all()
assert df[["title", "abstract"]].apply(
    lambda col: col.astype(str).str.strip().ne("").all()
).all()

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Departments:", df["department"].nunique())

display(df.head(3))

Dataset loaded successfully!
Shape: (103, 7)
Departments: 6


,project_id,title,abstract,department,year,keywords,source_page
0,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,Computer Science,2024 – 2025,"Moisturem Air humidity, AWTS system, Waterin, ...",5
1,P002,علامة CourseAssist: CLO Performance Assistant ...,The rapid evolution of educational technology ...,Computer Science,2024 – 2025,"Program Learning Outcomes, Course Learning Out...",6
2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,2024 – 2025,"Arabic Sign Language, Sign Language Gestures i...",7


## 3. Text Preparation

Combine project titles and abstracts into a single text representation while preserving semantic information.

In [6]:
# Prepare project texts
semantic_df = df.copy()

semantic_df["text"] = (
    semantic_df["title"].str.strip()
    + ". "
    + semantic_df["abstract"].str.strip()
)

semantic_df["text"] = (
    semantic_df["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

project_texts = semantic_df["text"].tolist()

print("Total project texts:", len(project_texts))
print("\nSample project text:")
print(project_texts[0][:500])

Total project texts: 103

Sample project text:
Automatic Plant Watering and Tracking System. Many people struggle with watering their plants regularly, which usually leads to plants becoming unhealthy or dying. To address this common issue, it is important to monitor basic factors such as soil moisture and temperature in order to determine the proper time and amount of water needed. Since manual watering is not always consistent or reliable, this automatic plant watering system is designed to assist gardeners by providing an accurate irrigat


## 4. Load Multilingual Sentence Transformer

Load a pretrained multilingual Sentence Transformer to generate semantic text embeddings for Arabic and English project descriptions.

In [7]:
# Load pretrained model
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(model_name, device=device)

print("Model loaded successfully!")
print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda
Embedding dimension: 768


/tmp/ipykernel_1757/400151206.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 5. Generate Project Embeddings

Encode the 103 graduation project descriptions into normalized dense vectors for semantic similarity retrieval.

In [8]:
# Generate embeddings
project_embeddings = model.encode(
    project_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Validate embeddings
assert project_embeddings.shape == (
    len(project_texts),
    model.get_sentence_embedding_dimension()
)

assert np.isfinite(project_embeddings).all()

print("Embeddings generated successfully!")
print("Embeddings shape:", project_embeddings.shape)
print("Data type:", project_embeddings.dtype)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings generated successfully!
Embeddings shape: (103, 768)
Data type: float32


/tmp/ipykernel_1757/265478390.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 6. Semantic Similarity and Top-K Retrieval

Retrieve the Top-K most semantically related graduation projects using Cosine Similarity.

In [9]:
def semantic_search(title, description, top_k=5):
    # Prepare query
    query_text = f"{title.strip()}. {description.strip()}"

    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    # Encode query
    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate similarity
    scores = cosine_similarity(
        query_embedding,
        project_embeddings
    )[0]

    # Retrieve Top-K
    top_indices = np.argsort(-scores)[:top_k]

    results = semantic_df.iloc[top_indices][
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["similarity_score"] = scores[top_indices]

    return results.reset_index(drop=True)


print("Semantic search function is ready!")

Semantic search function is ready!


## 7. Semantic Search Testing

Test semantic retrieval using an English project idea and inspect the Top-5 results.

In [10]:
# Test English query
test_title = "AI-Based Early Detection of Parkinson's Disease"

test_description = (
    "A mobile application that uses machine learning "
    "and wearable sensor data to detect early signs "
    "of Parkinson's disease and monitor patient symptoms."
)

english_results = semantic_search(
    title=test_title,
    description=test_description,
    top_k=5
)

display(
    english_results[
        ["rank", "project_id", "title", "similarity_score"]
    ].round(4)
)

,rank,project_id,title,similarity_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,0.8417
1,2,P080,A Smart Mobile Application For Skin Cancer Det...,0.5138
2,3,P089,AI-Powered Platform for Plant Identification a...,0.4956
3,4,P005,Thabat – ثبات,0.4779
4,5,P082,Predicting Diabetes Patients Based On Machine ...,0.4676


In [11]:
# Test Arabic query
arabic_title = "الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي"

arabic_description = (
    "تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار "
    "القابلة للارتداء للكشف المبكر عن مرض باركنسون "
    "ومراقبة أعراض المرض لدى المرضى."
)

arabic_results = semantic_search(
    title=arabic_title,
    description=arabic_description,
    top_k=5
)

display(
    arabic_results[
        ["rank", "project_id", "title", "similarity_score"]
    ].round(4)
)

,rank,project_id,title,similarity_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,0.6329
1,2,P082,Predicting Diabetes Patients Based On Machine ...,0.6151
2,3,P059,Blood Bank Inventory System - Artificial Intel...,0.5304
3,4,P089,AI-Powered Platform for Plant Identification a...,0.5104
4,5,P080,A Smart Mobile Application For Skin Cancer Det...,0.4995


## 8. Cross-Lingual Retrieval Analysis

Compare the Top-5 retrieved projects for equivalent English and Arabic queries.

In [13]:
comparison = pd.merge(
    english_results[["project_id", "rank", "similarity_score"]],
    arabic_results[["project_id", "rank", "similarity_score"]],
    on="project_id",
    how="outer",
    suffixes=("_en", "_ar")
)

comparison = comparison.sort_values(
    ["rank_en", "rank_ar"],
    na_position="last"
)

display(comparison.round(4))

overlap = len(
    set(english_results["project_id"])
    & set(arabic_results["project_id"])
)

print(f"Top-5 overlap: {overlap}/5")

,project_id,rank_en,similarity_score_en,rank_ar,similarity_score_ar
1,P025,1.0,0.8417,1.0,0.6329
3,P080,2.0,0.5138,5.0,0.4995
5,P089,3.0,0.4956,4.0,0.5104
0,P005,4.0,0.4779,NaN,NaN
4,P082,5.0,0.4676,2.0,0.6151
2,P059,NaN,NaN,3.0,0.5304


Top-5 overlap: 4/5


## 9. Retrieval Validation

Verify embedding normalization, similarity scores, and retrieval ranking.

In [14]:
# Validate embeddings
embedding_norms = np.linalg.norm(project_embeddings, axis=1)

assert np.allclose(embedding_norms, 1.0, atol=1e-5)

# Validate retrieval results
for results in [english_results, arabic_results]:
    assert len(results) == 5
    assert results["project_id"].is_unique
    assert results["similarity_score"].between(-1, 1).all()
    assert results["similarity_score"].is_monotonic_decreasing
    assert results["rank"].tolist() == [1, 2, 3, 4, 5]

print("All semantic retrieval validation checks passed!")

All semantic retrieval validation checks passed!


## 10. Save Results and Embeddings

Save the English and Arabic retrieval results and the generated project embeddings for future evaluation and reuse.

In [15]:
from pathlib import Path

# Create output directory
output_dir = Path("/content/origin_semantic_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save retrieval results
english_results.to_csv(
    output_dir / "semantic_english_results.csv",
    index=False,
    encoding="utf-8-sig"
)

arabic_results.to_csv(
    output_dir / "semantic_arabic_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save embeddings
np.save(
    output_dir / "project_embeddings.npy",
    project_embeddings
)

# Save project IDs
semantic_df[["project_id"]].to_csv(
    output_dir / "embedding_project_ids.csv",
    index=False
)

print("Files saved successfully!")

for file in sorted(output_dir.iterdir()):
    print(file.name)


Files saved successfully!
embedding_project_ids.csv
project_embeddings.npy
semantic_arabic_results.csv
semantic_english_results.csv


In [16]:
import shutil
from google.colab import files

# Create ZIP archive
archive_path = shutil.make_archive(
    "/content/origin_semantic_outputs",
    "zip",
    output_dir
)

files.download(archive_path)

print("Semantic search outputs downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Semantic search outputs downloaded successfully!


## 11. Conclusions

The multilingual Sentence Transformer successfully retrieved historical graduation projects using semantic similarity.

Initial English and Arabic tests ranked the most relevant Parkinson's disease detection project first. However, differences in ranking and similarity scores were observed between the two languages.

These preliminary results demonstrate the functionality of the semantic retrieval pipeline but do not establish its overall effectiveness. A final evaluation using human-annotated queries and retrieval metrics is required.